# Stage 4 – Data Preprocessing and Feature Engineering

## Water Point Operational Status Prediction

This notebook performs the preprocessing and feature engineering required to prepare the Tanzania waterpoint dataset for machine learning.

The preprocessing process is designed to:

- remove non-predictive and redundant features,
- control high-cardinality categorical variables,
- handle invalid and missing values,
- create meaningful engineered features,
- encode categorical variables,
- scale numerical variables where required,
- prevent data leakage,
- maintain a compact and interpretable feature space.

The main objective is to create a clean modelling dataset without unnecessarily generating hundreds of features.

## 1. Import Required Libraries

The required Python libraries are imported for data manipulation, visualization, train-test splitting, preprocessing, feature transformation, and saving the preprocessing pipeline.

In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import joblib

pd.set_option("display.max_columns", None)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load the Raw Datasets

The project uses two raw datasets:

- **Training Set Values** – contains the predictor variables describing each waterpoint.
- **Training Set Labels** – contains the target variable `status_group`.

The datasets are loaded directly from their raw CSV files so that preprocessing is performed independently from the Exploratory Data Analysis notebook.

In [44]:
values_path = "../data/raw/training_set_values.csv"
labels_path = "../data/raw/training_set_labels.csv"

df_values = pd.read_csv(values_path)
df_labels = pd.read_csv(labels_path)

print("Values dataset shape:", df_values.shape)
print("Labels dataset shape:", df_labels.shape)

Values dataset shape: (59400, 40)
Labels dataset shape: (59400, 2)


## 3. Merge Predictor and Target Data

The predictor dataset and target dataset are merged using the unique `id` field.

An inner join is used because each waterpoint in the predictor dataset is expected to have a corresponding target label.

In [45]:
df = pd.merge(
    df_values,
    df_labels,
    on="id",
    how="inner"
)

print("Merged dataset shape:", df.shape)

display(df.head())

Merged dataset shape: (59400, 41)


,id,amount_tsh,date_recorded,funder,gps_height,installer,longitude,latitude,wpt_name,num_private,basin,subvillage,region,region_code,district_code,lga,ward,population,public_meeting,recorded_by,scheme_management,scheme_name,permit,construction_year,extraction_type,extraction_type_group,extraction_type_class,management,management_group,payment,payment_type,water_quality,quality_group,quantity,quantity_group,source,source_type,source_class,waterpoint_type,waterpoint_type_group,status_group
0,69572,6000.0,2011-03-14,Roman,1390,Roman,34.938093,-9.856322,none,0,Lake Nyasa,Mnyusi B,Iringa,11,5,Ludewa,Mundindi,109,True,GeoData Consultants Ltd,VWC,Roman,False,1999,gravity,gravity,gravity,vwc,user-group,pay annually,annually,soft,good,enough,enough,spring,spring,groundwater,communal standpipe,communal standpipe,functional
1,8776,0.0,2013-03-06,Grumeti,1399,GRUMETI,34.698766,-2.147466,Zahanati,0,Lake Victoria,Nyamara,Mara,20,2,Serengeti,Natta,280,NaN,GeoData Consultants Ltd,Other,NaN,True,2010,gravity,gravity,gravity,wug,user-group,never pay,never pay,soft,good,insufficient,insufficient,rainwater harvesting,rainwater harvesting,surface,communal standpipe,communal standpipe,functional
2,34310,25.0,2013-02-25,Lottery Club,686,World vision,37.460664,-3.821329,Kwa Mahundi,0,Pangani,Majengo,Manyara,21,4,Simanjiro,Ngorika,250,True,GeoData Consultants Ltd,VWC,Nyumba ya mungu pipe scheme,True,2009,gravity,gravity,gravity,vwc,user-group,pay per bucket,per bucket,soft,good,enough,enough,dam,dam,surface,communal standpipe multiple,communal standpipe,functional
3,67743,0.0,2013-01-28,Unicef,263,UNICEF,38.486161,-11.155298,Zahanati Ya Nanyumbu,0,Ruvuma / Southern Coast,Mahakamani,Mtwara,90,63,Nanyumbu,Nanyumbu,58,True,GeoData Consultants Ltd,VWC,NaN,True,1986,submersible,submersible,submersible,vwc,user-group,never pay,never pay,soft,good,dry,dry,machine dbh,borehole,groundwater,communal standpipe multiple,communal standpipe,non functional
4,19728,0.0,2011-07-13,Action In A,0,Artisan,31.130847,-1.825359,Shuleni,0,Lake Victoria,Kyanyamisa,Kagera,18,1,Karagwe,Nyakasimbi,0,True,GeoData Consultants Ltd,NaN,NaN,True,0,gravity,gravity,gravity,other,other,never pay,never pay,soft,good,seasonal,seasonal,rainwater harvesting,rainwater harvesting,surface,communal standpipe,communal standpipe,functional


## 4. Separate Predictors and Target

The merged dataset is separated into:

- `X` – predictor variables used by the machine learning model.
- `y` – the target variable, `status_group`.

The target variable is kept separate from all preprocessing steps to prevent accidental data leakage.

In [46]:
X = df.drop(columns=["status_group"])
y = df["status_group"]

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:\n")
print(y.value_counts())

Predictor shape: (59400, 40)
Target shape: (59400,)

Target distribution:

status_group
functional                 32259
non functional             22824
functional needs repair     4317
Name: count, dtype: int64


In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

X_train shape: (47520, 40)
X_test shape : (11880, 40)
y_train shape: (47520,)
y_test shape : (11880,)


In [48]:
original_dist = (
    y.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

train_dist = (
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

test_dist = (
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

distribution_check = pd.DataFrame({
    "Original (%)": original_dist,
    "Training (%)": train_dist,
    "Testing (%)": test_dist
})

display(distribution_check)

,Original (%),Training (%),Testing (%)
status_group,,,
functional,54.31,54.31,54.31
non functional,38.42,38.42,38.43
functional needs repair,7.27,7.27,7.26


## 6. Initial Feature Selection

The original dataset contains 40 predictor variables. Using every available feature would introduce unnecessary redundancy, high-cardinality categorical variables, and a very large encoded feature space.

A compact feature-selection strategy is therefore applied before categorical encoding.

Features are excluded when they are:

- unique identifiers,
- constant or near-uninformative variables,
- extremely high-cardinality categorical variables,
- redundant representations of another retained feature,
- detailed geographic identifiers already represented by broader geographic features.

The objective is to retain a smaller set of meaningful and interpretable predictors while preventing unnecessary feature explosion during one-hot encoding.

### 6.1 Features Excluded from the Compact Model

The following feature groups are intentionally excluded.

**Identifiers and low-information variables**

- `id` – unique record identifier.
- `recorded_by` – constant feature.
- `num_private` – provides very limited useful variation.

**Very high-cardinality categorical variables**

- `wpt_name`
- `subvillage`
- `scheme_name`
- `funder`
- `installer`
- `ward`
- `lga`

These variables contain a large number of unique categories and would create an unnecessarily large number of one-hot encoded features.

**Redundant hierarchical features**

Only one representative variable is retained from feature groups that describe the same underlying concept at different levels.

Examples include extraction type, management, payment, water quality, quantity, source, and waterpoint type.

**Redundant geographic representations**

Broader geographic information such as `region`, `basin`, latitude, and longitude is retained, while additional coded geographic variables such as `region_code` and `district_code` are excluded.

This approach keeps the feature set compact, interpretable, and suitable for the final prediction application.

### 6.2 Selected Raw Predictors

A compact set of numerical, temporal, geographic, operational, and management-related variables is retained.

At this stage, `date_recorded` and `construction_year` are retained temporarily because they will later be used to engineer the more meaningful `waterpoint_age` feature.

In [49]:
selected_features = [
    # Numerical / temporal
    "amount_tsh",
    "gps_height",
    "longitude",
    "latitude",
    "population",
    "construction_year",
    "date_recorded",

    # Geographic
    "basin",
    "region",

    # Management / administrative
    "public_meeting",
    "scheme_management",
    "permit",
    "management_group",
    "payment_type",

    # Waterpoint characteristics
    "extraction_type_class",
    "quality_group",
    "quantity",
    "source_type",
    "waterpoint_type_group"
]

print("Number of selected raw features:", len(selected_features))
print("\nSelected features:")

for feature in selected_features:
    print("-", feature)

Number of selected raw features: 19

Selected features:
- amount_tsh
- gps_height
- longitude
- latitude
- population
- construction_year
- date_recorded
- basin
- region
- public_meeting
- scheme_management
- permit
- management_group
- payment_type
- extraction_type_class
- quality_group
- quantity
- source_type
- waterpoint_type_group


In [50]:
excluded_features = [
    col for col in X_train.columns
    if col not in selected_features
]

print("Original predictor count:", X_train.shape[1])
print("Selected predictor count:", len(selected_features))
print("Excluded predictor count:", len(excluded_features))

print("\nExcluded features:")
for feature in excluded_features:
    print("-", feature)

Original predictor count: 40
Selected predictor count: 19
Excluded predictor count: 21

Excluded features:
- id
- funder
- installer
- wpt_name
- num_private
- subvillage
- region_code
- district_code
- lga
- ward
- recorded_by
- scheme_name
- extraction_type
- extraction_type_group
- management
- payment
- water_quality
- quantity_group
- source
- source_class
- waterpoint_type


### 6.3 Create Compact Training and Testing Sets

The same selected predictor list is applied to both the training and testing datasets.

Only the training dataset will later be used to fit preprocessing operations such as imputation, encoding, and scaling.

In [51]:
X_train_selected = X_train[selected_features].copy()
X_test_selected = X_test[selected_features].copy()

print("Selected training shape:", X_train_selected.shape)
print("Selected testing shape :", X_test_selected.shape)

Selected training shape: (47520, 19)
Selected testing shape : (11880, 19)


In [52]:
display(X_train_selected.head())

print("\nData types:")
print(X_train_selected.dtypes)

,amount_tsh,gps_height,longitude,latitude,population,construction_year,date_recorded,basin,region,public_meeting,scheme_management,permit,management_group,payment_type,extraction_type_class,quality_group,quantity,source_type,waterpoint_type_group
43360,0.0,0,33.542898,-9.174777,0,0,2011-07-27,Lake Nyasa,Mbeya,True,VWC,NaN,user-group,never pay,gravity,good,insufficient,spring,communal standpipe
7263,500.0,2049,34.665760,-9.308548,175,2008,2011-03-23,Rufiji,Iringa,True,WUA,True,user-group,monthly,gravity,good,enough,spring,communal standpipe
2486,25.0,290,38.238568,-6.179919,2300,2010,2011-03-07,Wami / Ruvu,Pwani,True,VWC,False,user-group,per bucket,handpump,salty,insufficient,shallow well,hand pump
313,0.0,0,30.716727,-1.289055,0,0,2011-07-31,Lake Victoria,Kagera,True,NaN,True,user-group,never pay,other,good,enough,shallow well,other
52726,0.0,0,35.389331,-6.399942,0,0,2011-03-10,Internal,Dodoma,True,VWC,True,user-group,per bucket,motorpump,good,enough,borehole,communal standpipe



Data types:
amount_tsh               float64
gps_height                 int64
longitude                float64
latitude                 float64
population                 int64
construction_year          int64
date_recorded                str
basin                        str
region                       str
public_meeting            object
scheme_management            str
permit                    object
management_group             str
payment_type                 str
extraction_type_class        str
quality_group                str
quantity                     str
source_type                  str
waterpoint_type_group        str
dtype: object


In [53]:
categorical_check = [
    "basin",
    "region",
    "public_meeting",
    "scheme_management",
    "permit",
    "management_group",
    "payment_type",
    "extraction_type_class",
    "quality_group",
    "quantity",
    "source_type",
    "waterpoint_type_group"
]

cardinality_table = pd.DataFrame({
    "Feature": categorical_check,
    "Unique_Values": [
        X_train_selected[col].nunique(dropna=False)
        for col in categorical_check
    ]
}).sort_values("Unique_Values", ascending=False)

display(cardinality_table)

,Feature,Unique_Values
1,region,21
3,scheme_management,12
0,basin,9
7,extraction_type_class,7
10,source_type,7
6,payment_type,7
11,waterpoint_type_group,6
8,quality_group,6
9,quantity,5
5,management_group,5


In [54]:
X_train_selected["date_recorded"] = pd.to_datetime(
    X_train_selected["date_recorded"],
    errors="coerce"
)

X_test_selected["date_recorded"] = pd.to_datetime(
    X_test_selected["date_recorded"],
    errors="coerce"
)

print("Training date type:", X_train_selected["date_recorded"].dtype)
print("Testing date type :", X_test_selected["date_recorded"].dtype)

print("\nMissing dates in training:",
      X_train_selected["date_recorded"].isna().sum())

print("Missing dates in testing:",
      X_test_selected["date_recorded"].isna().sum())

Training date type: datetime64[us]
Testing date type : datetime64[us]

Missing dates in training: 0
Missing dates in testing: 0


In [55]:
X_train_selected["record_year"] = (
    X_train_selected["date_recorded"].dt.year
)

X_test_selected["record_year"] = (
    X_test_selected["date_recorded"].dt.year
)

print("Training recording years:")
print(
    X_train_selected["record_year"]
    .value_counts()
    .sort_index()
)

Training recording years:
record_year
2002        1
2004       25
2011    22928
2012     5139
2013    19427
Name: count, dtype: int64


In [56]:
train_zero_year = (
    X_train_selected["construction_year"] == 0
).sum()

test_zero_year = (
    X_test_selected["construction_year"] == 0
).sum()

train_future_year = (
    (X_train_selected["construction_year"] > 0) &
    (
        X_train_selected["construction_year"]
        > X_train_selected["record_year"]
    )
).sum()

test_future_year = (
    (X_test_selected["construction_year"] > 0) &
    (
        X_test_selected["construction_year"]
        > X_test_selected["record_year"]
    )
).sum()

print("Training construction_year = 0 :", train_zero_year)
print("Testing construction_year = 0  :", test_zero_year)

print("\nTraining construction year > record year:",
      train_future_year)

print("Testing construction year > record year :",
      test_future_year)

Training construction_year = 0 : 16517
Testing construction_year = 0  : 4192

Training construction year > record year: 9
Testing construction year > record year : 0


In [57]:
# Invalid zero years
X_train_selected.loc[
    X_train_selected["construction_year"] == 0,
    "construction_year"
] = np.nan

X_test_selected.loc[
    X_test_selected["construction_year"] == 0,
    "construction_year"
] = np.nan


# Construction year cannot be later than observation year
X_train_selected.loc[
    X_train_selected["construction_year"]
    > X_train_selected["record_year"],
    "construction_year"
] = np.nan

X_test_selected.loc[
    X_test_selected["construction_year"]
    > X_test_selected["record_year"],
    "construction_year"
] = np.nan


print(
    "Missing construction years in training:",
    X_train_selected["construction_year"].isna().sum()
)

print(
    "Missing construction years in testing:",
    X_test_selected["construction_year"].isna().sum()
)

Missing construction years in training: 16526
Missing construction years in testing: 4192


In [58]:
invalid_train_coords = (
    (X_train_selected["longitude"] == 0) &
    (X_train_selected["latitude"].abs() < 0.001)
)

invalid_test_coords = (
    (X_test_selected["longitude"] == 0) &
    (X_test_selected["latitude"].abs() < 0.001)
)

print(
    "Invalid coordinate records in training:",
    invalid_train_coords.sum()
)

print(
    "Invalid coordinate records in testing:",
    invalid_test_coords.sum()
)

Invalid coordinate records in training: 1442
Invalid coordinate records in testing: 370


In [59]:
X_train_selected.loc[
    invalid_train_coords,
    ["longitude", "latitude"]
] = np.nan

X_test_selected.loc[
    invalid_test_coords,
    ["longitude", "latitude"]
] = np.nan

print(
    "Missing longitude values in training:",
    X_train_selected["longitude"].isna().sum()
)

print(
    "Missing latitude values in training:",
    X_train_selected["latitude"].isna().sum()
)

Missing longitude values in training: 1442
Missing latitude values in training: 1442


### 7.5 Treatment of Other Zero Values

Zero values in variables such as `population`, `gps_height`, and `amount_tsh` are not automatically converted to missing values.

Unlike the invalid construction-year and coordinate patterns, the available dataset documentation does not establish that every zero in these variables represents missing data.

Therefore, these values are retained to avoid making unsupported assumptions or unnecessarily modifying valid observations.

## 8. Feature Engineering

### 8.1 Waterpoint Age

A new feature named `waterpoint_age` is created using the construction year and the year in which the waterpoint was observed.

\[
\text{Waterpoint Age} =
\text{Record Year} -
\text{Construction Year}
\]

Waterpoint age provides a more directly interpretable representation of infrastructure age than construction year alone and may capture deterioration associated with older water infrastructure.

Only this meaningful age feature is created rather than generating many unnecessary derived variables.

In [60]:
X_train_selected["waterpoint_age"] = (
    X_train_selected["record_year"]
    - X_train_selected["construction_year"]
)

X_test_selected["waterpoint_age"] = (
    X_test_selected["record_year"]
    - X_test_selected["construction_year"]
)

print(
    X_train_selected["waterpoint_age"]
    .describe()
)

count    30994.000000
mean        15.348906
std         12.521597
min          0.000000
25%          4.000000
50%         13.000000
75%         25.000000
max         53.000000
Name: waterpoint_age, dtype: float64


In [61]:
print(
    "Negative ages in training:",
    (X_train_selected["waterpoint_age"] < 0).sum()
)

print(
    "Negative ages in testing:",
    (X_test_selected["waterpoint_age"] < 0).sum()
)

Negative ages in training: 0
Negative ages in testing: 0


### 8.2 Remove Temporary and Redundant Date Features

After calculating `waterpoint_age`, the original `date_recorded`, `record_year`, and `construction_year` variables are removed from the modelling dataset.

This reduces redundancy and keeps the final feature set compact.

In [62]:
columns_to_remove = [
    "date_recorded",
    "record_year",
    "construction_year"
]

X_train_selected = X_train_selected.drop(
    columns=columns_to_remove
)

X_test_selected = X_test_selected.drop(
    columns=columns_to_remove
)

print("Training shape:", X_train_selected.shape)
print("Testing shape :", X_test_selected.shape)

print("\nRemaining features:")
print(X_train_selected.columns.tolist())

Training shape: (47520, 18)
Testing shape : (11880, 18)

Remaining features:
['amount_tsh', 'gps_height', 'longitude', 'latitude', 'population', 'basin', 'region', 'public_meeting', 'scheme_management', 'permit', 'management_group', 'payment_type', 'extraction_type_class', 'quality_group', 'quantity', 'source_type', 'waterpoint_type_group', 'waterpoint_age']


## 9. Define Numerical and Categorical Features

After feature selection, invalid-value treatment, and feature engineering, the remaining predictors are divided into numerical and categorical groups.

This separation allows appropriate preprocessing techniques to be applied to each feature type.

- Numerical variables will use median imputation and standardisation.
- Categorical variables will use explicit missing-value handling and one-hot encoding.

In [63]:
numerical_features = [
    "amount_tsh",
    "gps_height",
    "longitude",
    "latitude",
    "population",
    "waterpoint_age"
]

categorical_features = [
    "basin",
    "region",
    "public_meeting",
    "scheme_management",
    "permit",
    "management_group",
    "payment_type",
    "extraction_type_class",
    "quality_group",
    "quantity",
    "source_type",
    "waterpoint_type_group"
]

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Total features:", len(numerical_features) + len(categorical_features))

Numerical features: 6
Categorical features: 12
Total features: 18


In [64]:
expected_features = numerical_features + categorical_features

missing_columns = [
    col for col in expected_features
    if col not in X_train_selected.columns
]

extra_columns = [
    col for col in X_train_selected.columns
    if col not in expected_features
]

print("Missing expected columns:", missing_columns)
print("Unexpected extra columns:", extra_columns)

Missing expected columns: []
Unexpected extra columns: []


## 10. Missing-Value Analysis After Cleaning

Missing values are re-examined after invalid-value treatment and feature engineering.

This check is performed separately on the training data before defining the preprocessing pipeline.

Missing values are not manually filled at this stage. Instead, imputation will be incorporated into the preprocessing pipeline so that the required statistics are learned only from the training data.

In [65]:
missing_summary = pd.DataFrame({
    "Missing_Count": X_train_selected.isna().sum(),
    "Missing_Percentage": (
        X_train_selected.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    "Missing_Percentage",
    ascending=False
)

display(missing_summary)

,Missing_Count,Missing_Percentage
waterpoint_age,16526,34.78
scheme_management,3129,6.58
public_meeting,2644,5.56
permit,2443,5.14
latitude,1442,3.03
longitude,1442,3.03


### 10.1 Standardise Boolean Categorical Variables

The `public_meeting` and `permit` variables contain Boolean values together with missing values.

For consistent categorical processing, Boolean values are converted into the text categories `Yes` and `No`. Missing values remain missing and will later be handled by the categorical imputation pipeline.

In [66]:
boolean_categorical_features = [
    "public_meeting",
    "permit"
]

boolean_mapping = {
    True: "Yes",
    False: "No"
}

for col in boolean_categorical_features:
    X_train_selected[col] = (
        X_train_selected[col]
        .map(boolean_mapping)
    )

    X_test_selected[col] = (
        X_test_selected[col]
        .map(boolean_mapping)
    )

print("public_meeting values:")
print(X_train_selected["public_meeting"].value_counts(dropna=False))

print("\npermit values:")
print(X_train_selected["permit"].value_counts(dropna=False))

public_meeting values:
public_meeting
Yes    40838
No      4038
NaN     2644
Name: count, dtype: int64

permit values:
permit
Yes    31071
No     14006
NaN     2443
Name: count, dtype: int64


In [67]:
print("Categorical feature data types:\n")

for col in categorical_features:
    print(
        f"{col:25s}",
        X_train_selected[col].dtype
    )

Categorical feature data types:

basin                     str
region                    str
public_meeting            str
scheme_management         str
permit                    str
management_group          str
payment_type              str
extraction_type_class     str
quality_group             str
quantity                  str
source_type               str
waterpoint_type_group     str


## 11. Numerical Preprocessing

Numerical missing values are handled using median imputation.

Median imputation is preferred because several numerical variables contain skewed distributions and extreme observations, making the median less sensitive to unusually large values than the mean.

After imputation, numerical variables are standardised using `StandardScaler`. This creates a common scale that is suitable for algorithms that are sensitive to differences in feature magnitude.

In [68]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

print(numerical_pipeline)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


## 12. Categorical Preprocessing

Missing categorical values are replaced with an explicit `Missing` category rather than automatically assigning the most frequent category.

This preserves the information that a value was unavailable.

The retained categorical variables have deliberately low cardinality. They are encoded using one-hot encoding.

`handle_unknown="ignore"` ensures that an unseen category in the test set or future application input does not cause the prediction system to fail.

The first category of each variable is dropped to reduce unnecessary dummy-variable redundancy.

In [69]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ]
)

print(categorical_pipeline)

Pipeline(steps=[('imputer',
                 SimpleImputer(fill_value='Missing', strategy='constant')),
                ('encoder',
                 OneHotEncoder(drop='first', handle_unknown='ignore'))])


## 13. Build the Preprocessing Transformer

The numerical and categorical preprocessing pipelines are combined using a `ColumnTransformer`.

This provides a single reusable preprocessing object that can later be fitted on the training data and used consistently for:

- model training,
- test-set transformation,
- model evaluation,
- backend prediction,
- future frontend inputs.

Keeping preprocessing inside a reusable pipeline also helps prevent differences between model-development preprocessing and application-time preprocessing.

In [70]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['amount_tsh', 'gps_height', 'longitude',
                                  'latitude', 'population', 'waterpoint_age']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Missing',
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'))]),
                                 ['basin', 'region', 'public_meeting',
 

## 14. Fit the Preprocessing Transformer

The preprocessing transformer is fitted using only the training dataset.

During fitting:

- numerical imputation values are learned from the training data,
- scaling parameters are calculated from the training data,
- categorical levels are learned from the training data.

The test dataset is not used to learn any preprocessing parameters. This prevents information leakage from the test set into model development.

In [71]:
X_train_processed = preprocessor.fit_transform(
    X_train_selected
)

X_test_processed = preprocessor.transform(
    X_test_selected
)

print("Training data before preprocessing:",
      X_train_selected.shape)

print("Testing data before preprocessing :",
      X_test_selected.shape)

print("\nTraining data after preprocessing:",
      X_train_processed.shape)

print("Testing data after preprocessing :",
      X_test_processed.shape)

Training data before preprocessing: (47520, 18)
Testing data before preprocessing : (11880, 18)

Training data after preprocessing: (47520, 85)
Testing data after preprocessing : (11880, 85)


### 14.1 Inspect the Final Model Features

After preprocessing, the final transformed feature names are inspected.

Although the prediction application will use the original human-readable inputs, one-hot encoding internally expands categorical variables into binary model features.

The final feature count is checked to ensure that the preprocessing strategy has maintained a compact feature space.

In [72]:
feature_names = preprocessor.get_feature_names_out()

print("Total final model features:",
      len(feature_names))

print("\nFinal feature names:")

for i, feature in enumerate(feature_names, start=1):
    print(f"{i:>3}. {feature}")

Total final model features: 85

Final feature names:
  1. num__amount_tsh
  2. num__gps_height
  3. num__longitude
  4. num__latitude
  5. num__population
  6. num__waterpoint_age
  7. cat__basin_Lake Nyasa
  8. cat__basin_Lake Rukwa
  9. cat__basin_Lake Tanganyika
 10. cat__basin_Lake Victoria
 11. cat__basin_Pangani
 12. cat__basin_Rufiji
 13. cat__basin_Ruvuma / Southern Coast
 14. cat__basin_Wami / Ruvu
 15. cat__region_Dar es Salaam
 16. cat__region_Dodoma
 17. cat__region_Iringa
 18. cat__region_Kagera
 19. cat__region_Kigoma
 20. cat__region_Kilimanjaro
 21. cat__region_Lindi
 22. cat__region_Manyara
 23. cat__region_Mara
 24. cat__region_Mbeya
 25. cat__region_Morogoro
 26. cat__region_Mtwara
 27. cat__region_Mwanza
 28. cat__region_Pwani
 29. cat__region_Rukwa
 30. cat__region_Ruvuma
 31. cat__region_Shinyanga
 32. cat__region_Singida
 33. cat__region_Tabora
 34. cat__region_Tanga
 35. cat__public_meeting_No
 36. cat__public_meeting_Yes
 37. cat__scheme_management_Missing
 38.

In [73]:
numerical_output_features = [
    feature for feature in feature_names
    if feature.startswith("num__")
]

categorical_output_features = [
    feature for feature in feature_names
    if feature.startswith("cat__")
]

print("Numerical output features:",
      len(numerical_output_features))

print("Encoded categorical features:",
      len(categorical_output_features))

print("Total:",
      len(feature_names))

Numerical output features: 6
Encoded categorical features: 79
Total: 85


### 14.2 Validate the Processed Data

The processed training and testing matrices are checked for missing and infinite values.

A valid modelling dataset should contain no unresolved missing values or infinite numerical values after preprocessing.

In [74]:
def check_processed_matrix(matrix, name):
    # Handles both sparse and dense matrices
    if hasattr(matrix, "toarray"):
        values = matrix.data
    else:
        values = np.asarray(matrix)

    missing_count = np.isnan(values).sum()
    infinite_count = np.isinf(values).sum()

    print(f"{name}")
    print("Missing values :", missing_count)
    print("Infinite values:", infinite_count)
    print()


check_processed_matrix(
    X_train_processed,
    "Processed Training Data"
)

check_processed_matrix(
    X_test_processed,
    "Processed Testing Data"
)

Processed Training Data
Missing values : 0
Infinite values: 0

Processed Testing Data
Missing values : 0
Infinite values: 0



In [75]:
assert (
    X_train_processed.shape[1]
    == X_test_processed.shape[1]
), "Train and test feature counts do not match."

assert (
    X_train_processed.shape[1]
    == len(feature_names)
), "Feature-name count does not match transformed data."

print("Feature dimension verification passed.")
print(
    "Both datasets contain",
    X_train_processed.shape[1],
    "model features."
)

Feature dimension verification passed.
Both datasets contain 85 model features.


## 15. Create a Reusable Feature-Engineering Function

The preprocessing performed before numerical and categorical transformation must also be reproducible when new waterpoint records are provided to the final prediction system.

A reusable feature-engineering function is therefore created to:

- select the required raw predictors,
- convert the recorded date,
- identify invalid construction years,
- identify invalid geographic coordinates,
- calculate `waterpoint_age`,
- standardise Boolean categorical values,
- remove temporary date-related variables.

This ensures that training data, test data, and future application inputs follow the same feature-engineering rules.

In [76]:
def prepare_model_features(data):
    """
    Convert raw waterpoint records into the 18 selected model-level features.
    """

    df_prepared = data.copy()

    # Raw columns required before feature engineering
    required_raw_features = [
        "amount_tsh",
        "gps_height",
        "longitude",
        "latitude",
        "population",
        "construction_year",
        "date_recorded",
        "basin",
        "region",
        "public_meeting",
        "scheme_management",
        "permit",
        "management_group",
        "payment_type",
        "extraction_type_class",
        "quality_group",
        "quantity",
        "source_type",
        "waterpoint_type_group"
    ]

    # Keep only required raw predictors
    df_prepared = df_prepared[required_raw_features].copy()

    # Convert date
    df_prepared["date_recorded"] = pd.to_datetime(
        df_prepared["date_recorded"],
        errors="coerce"
    )

    # Extract temporary observation year
    df_prepared["record_year"] = (
        df_prepared["date_recorded"].dt.year
    )

    # Invalid construction year: zero
    df_prepared.loc[
        df_prepared["construction_year"] == 0,
        "construction_year"
    ] = np.nan

    # Invalid construction year: later than observation year
    df_prepared.loc[
        df_prepared["construction_year"]
        > df_prepared["record_year"],
        "construction_year"
    ] = np.nan

    # Detect invalid coordinate pattern
    invalid_coords = (
        (df_prepared["longitude"] == 0) &
        (df_prepared["latitude"].abs() < 0.001)
    )

    df_prepared.loc[
        invalid_coords,
        ["longitude", "latitude"]
    ] = np.nan

    # Create engineered age feature
    df_prepared["waterpoint_age"] = (
        df_prepared["record_year"]
        - df_prepared["construction_year"]
    )

    # Standardise Boolean categorical features
    boolean_mapping = {
        True: "Yes",
        False: "No"
    }

    for col in ["public_meeting", "permit"]:
        df_prepared[col] = (
            df_prepared[col]
            .map(boolean_mapping)
        )

    # Remove temporary/raw temporal fields
    df_prepared = df_prepared.drop(
        columns=[
            "date_recorded",
            "record_year",
            "construction_year"
        ]
    )

    return df_prepared

### 15.1 Validate the Reusable Feature-Engineering Function

The reusable function is applied to the original training and testing predictors.

Its output is checked to ensure that it produces the same 18 model-level predictors used during the manual preprocessing process.

In [77]:
X_train_prepared_check = prepare_model_features(X_train)
X_test_prepared_check = prepare_model_features(X_test)

print("Prepared training shape:",
      X_train_prepared_check.shape)

print("Prepared testing shape :",
      X_test_prepared_check.shape)

print("\nPrepared columns:")
print(X_train_prepared_check.columns.tolist())

Prepared training shape: (47520, 18)
Prepared testing shape : (11880, 18)

Prepared columns:
['amount_tsh', 'gps_height', 'longitude', 'latitude', 'population', 'basin', 'region', 'public_meeting', 'scheme_management', 'permit', 'management_group', 'payment_type', 'extraction_type_class', 'quality_group', 'quantity', 'source_type', 'waterpoint_type_group', 'waterpoint_age']


In [78]:
print(
    "Training columns match:",
    list(X_train_prepared_check.columns)
    == list(X_train_selected.columns)
)

print(
    "Testing columns match:",
    list(X_test_prepared_check.columns)
    == list(X_test_selected.columns)
)

Training columns match: True
Testing columns match: True


In [79]:
training_match = (
    X_train_prepared_check
    .reset_index(drop=True)
    .equals(
        X_train_selected
        .reset_index(drop=True)
    )
)

testing_match = (
    X_test_prepared_check
    .reset_index(drop=True)
    .equals(
        X_test_selected
        .reset_index(drop=True)
    )
)

print("Training feature values match:", training_match)
print("Testing feature values match :", testing_match)

Training feature values match: True
Testing feature values match : True


In [80]:
X_train_pipeline_check = preprocessor.transform(
    X_train_prepared_check
)

X_test_pipeline_check = preprocessor.transform(
    X_test_prepared_check
)

print(
    "Training transformed shape:",
    X_train_pipeline_check.shape
)

print(
    "Testing transformed shape :",
    X_test_pipeline_check.shape
)

Training transformed shape: (47520, 85)
Testing transformed shape : (11880, 85)


## 16. Data Leakage Verification

The preprocessing workflow was reviewed to ensure that information from the test dataset was not used during model preparation.

The following controls were applied:

- The target variable `status_group` was separated before preprocessing.
- The train-test split was performed before learned preprocessing operations.
- Stratification was used to preserve target-class proportions.
- Numerical imputation statistics were learned only from the training data.
- Scaling parameters were learned only from the training data.
- Categorical levels for one-hot encoding were learned only from the training data.
- The test dataset was transformed using the already-fitted training preprocessor.
- No class-balancing technique was applied before the train-test split.
- Target information was not used to create predictor features.
- Unique identifiers were excluded from the final predictor set.

These controls reduce the risk of data leakage and provide a more reliable evaluation of future machine-learning models.

In [81]:
# Check target is not present in model inputs
assert "status_group" not in X_train_selected.columns
assert "status_group" not in X_test_selected.columns

# Check ID is not present
assert "id" not in X_train_selected.columns
assert "id" not in X_test_selected.columns

# Check train/test contain the same raw model features
assert list(X_train_selected.columns) == list(X_test_selected.columns)

# Check processed dimensions match
assert X_train_processed.shape[1] == X_test_processed.shape[1]

# Check feature names match processed feature count
assert X_train_processed.shape[1] == len(feature_names)

print("All leakage and feature-consistency checks passed.")

All leakage and feature-consistency checks passed.


## 17. Final Feature Set

The original dataset contained 40 predictor variables.

After feature selection and feature engineering, 18 meaningful predictor variables were retained before encoding.

### Numerical Features

1. `amount_tsh`
2. `gps_height`
3. `longitude`
4. `latitude`
5. `population`
6. `waterpoint_age`

### Categorical Features

1. `basin`
2. `region`
3. `public_meeting`
4. `scheme_management`
5. `permit`
6. `management_group`
7. `payment_type`
8. `extraction_type_class`
9. `quality_group`
10. `quantity`
11. `source_type`
12. `waterpoint_type_group`

One-hot encoding expands the categorical variables internally for machine-learning algorithms. However, the prediction system still requires only the original human-readable input variables rather than the encoded columns.

## 18. Summary of Preprocessing Decisions

| Issue | Decision | Reason |
|---|---|---|
| Unique identifier | Removed `id` | Identifier does not represent a meaningful predictive characteristic |
| Constant / low-information variables | Removed | Provide little useful predictive information |
| Extremely high-cardinality variables | Removed | Prevent excessive one-hot encoded feature expansion |
| Redundant hierarchical variables | Retained one representative feature | Reduce duplication and dimensionality |
| Invalid `construction_year = 0` | Converted to missing | Zero is not a valid construction year |
| Construction year after observation year | Converted to missing | Logically inconsistent |
| Invalid zero coordinates | Converted to missing | Do not represent valid Tanzania locations |
| Other numerical zeros | Retained | Insufficient evidence that all represent missing data |
| Missing numerical values | Median imputation | More robust to skewed data and extreme values |
| Missing categorical values | Explicit `Missing` category | Preserves information that the original value was unavailable |
| Infrastructure age | Created `waterpoint_age` | More directly represents ageing of the waterpoint |
| Categorical encoding | One-hot encoding | Suitable for nominal categories without natural ordering |
| Numerical scaling | StandardScaler | Supports scale-sensitive machine-learning algorithms |
| Train-test split | 80:20 stratified split | Preserves class proportions |
| Class imbalance | Not modified during preprocessing | Balancing techniques will be evaluated only on training data during modelling |

In [82]:
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

preprocessor_path = models_dir / "preprocessor.joblib"

joblib.dump(
    preprocessor,
    preprocessor_path
)

print("Preprocessor saved to:")
print(preprocessor_path)

Preprocessor saved to:
..\models\preprocessor.joblib


In [83]:
feature_names_df = pd.DataFrame({
    "feature_name": feature_names
})

feature_names_path = models_dir / "processed_feature_names.csv"

feature_names_df.to_csv(
    feature_names_path,
    index=False
)

print("Feature names saved to:")
print(feature_names_path)

print("\nTotal saved feature names:",
      len(feature_names_df))

Feature names saved to:
..\models\processed_feature_names.csv

Total saved feature names: 85


## 19. Stage 4 Summary

The preprocessing and feature-engineering stage transformed the original waterpoint dataset into a compact and reproducible modelling representation.

The original dataset contained 40 predictor variables. Feature selection was used to remove identifiers, low-information variables, highly cardinal categorical variables, redundant geographic variables, and duplicate hierarchical representations.

Invalid construction years and invalid geographic coordinates were identified and converted to missing values rather than deleting complete waterpoint observations.

A meaningful engineered feature, `waterpoint_age`, was created from the construction year and observation date.

The final pre-encoding dataset contains:

- **6 numerical predictors**
- **12 categorical predictors**
- **18 total model-level predictors**

Numerical missing values are handled using median imputation and numerical variables are standardised. Categorical missing values are represented explicitly and nominal categorical variables are one-hot encoded.

After transformation, the modelling dataset contains **85 internal machine-learning features**. This dimensionality is controlled and substantially smaller than would result from encoding the original high-cardinality variables.

All learned preprocessing operations were fitted using only the training dataset. The test dataset remained independent and was transformed using the fitted preprocessing pipeline to reduce the risk of information leakage.

The fitted preprocessing transformer was saved for reuse during model development and final system integration.

Class imbalance was not modified during this stage. Techniques such as class weighting and SMOTE/SMOTENC may be investigated during model development using the training data only.

## 20. Save Pre-Encoding Modelling Data

The training and testing datasets are saved before numerical imputation, scaling, and one-hot encoding.

This allows preprocessing to be fitted independently within cross-validation folds during model development, reducing the risk of validation leakage.

In [84]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

train_model_data = X_train_selected.copy()
train_model_data["status_group"] = y_train

test_model_data = X_test_selected.copy()
test_model_data["status_group"] = y_test

train_path = processed_dir / "train_model_data.csv"
test_path = processed_dir / "test_model_data.csv"

train_model_data.to_csv(train_path, index=False)
test_model_data.to_csv(test_path, index=False)

print("Training modelling data:", train_model_data.shape)
print("Testing modelling data :", test_model_data.shape)

print("\nSaved:")
print(train_path)
print(test_path)

Training modelling data: (47520, 19)
Testing modelling data : (11880, 19)

Saved:
..\data\processed\train_model_data.csv
..\data\processed\test_model_data.csv
